# 04: Scheme Performance — Data Cleaning & Pre-processing

**Day 2 | Part 3 — Bluestock Mutual Fund Analytics**

Pipeline steps:
1. Load raw `07_scheme_performance.csv`
2. Validate all return columns are numeric
3. Flag schemes with negative Sharpe ratios
4. Validate & flag expense_ratio outside 0.1% – 2.5% range
5. Remove duplicates
6. Save `data/processed/clean_performance.csv`
7. Verification summary

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW_PATH      = Path('../data/raw/07_scheme_performance.csv')
PROCESSED_DIR = Path('../data/processed')
OUTPUT_PATH   = PROCESSED_DIR / 'clean_performance.csv'

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

## Step 1 — Load Raw Data

In [2]:
df_raw = pd.read_csv(RAW_PATH)

print(f"Raw shape   : {df_raw.shape}")
print(f"Columns     : {df_raw.columns.tolist()}")
print(f"\nDtypes:\n{df_raw.dtypes}\n")
df_raw.head(3)

Raw shape   : (40, 19)
Columns     : ['amfi_code', 'scheme_name', 'fund_house', 'category', 'plan', 'return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct', 'benchmark_3yr_pct', 'alpha', 'beta', 'sharpe_ratio', 'sortino_ratio', 'std_dev_ann_pct', 'max_drawdown_pct', 'aum_crore', 'expense_ratio_pct', 'morningstar_rating', 'risk_grade']

Dtypes:
amfi_code               int64
scheme_name            object
fund_house             object
category               object
plan                   object
return_1yr_pct        float64
return_3yr_pct        float64
return_5yr_pct        float64
benchmark_3yr_pct     float64
alpha                 float64
beta                  float64
sharpe_ratio          float64
sortino_ratio         float64
std_dev_ann_pct       float64
max_drawdown_pct      float64
aum_crore               int64
expense_ratio_pct     float64
morningstar_rating      int64
risk_grade             object
dtype: object



,amfi_code,scheme_name,fund_house,category,plan,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,aum_crore,expense_ratio_pct,morningstar_rating,risk_grade
0,119551,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,Large Cap,Regular,12.42,12.36,14.45,11.49,0.87,0.89,0.88,1.29,14.0,-21.70,14288,1.54,4,Moderate
1,119552,SBI Bluechip Fund - Direct Plan - Growth,SBI Mutual Fund,Large Cap,Direct,15.25,11.30,14.23,9.52,1.78,0.87,0.81,1.29,14.0,-24.43,1231,0.66,3,Moderate
2,119598,SBI Small Cap Fund - Regular Plan - Growth,SBI Mutual Fund,Small Cap,Regular,24.56,23.39,20.67,22.16,1.23,0.89,0.94,1.35,25.0,-13.35,19259,1.43,5,Very High


## Step 2 — Validate Return Values are Numeric

In [3]:
df = df_raw.copy()

RETURN_COLS = [
    'return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct',
    'benchmark_3yr_pct', 'alpha', 'beta', 'sharpe_ratio',
    'sortino_ratio', 'std_dev_ann_pct', 'max_drawdown_pct',
    'aum_crore', 'expense_ratio_pct'
]

print("Numeric validation for return/metric columns:")
coerce_issues = 0
for col in RETURN_COLS:
    # Coerce any non-numeric values
    original_nulls = df[col].isna().sum()
    df[col] = pd.to_numeric(df[col], errors='coerce')
    new_nulls = df[col].isna().sum()
    coerced = new_nulls - original_nulls
    if coerced > 0:
        print(f"  ⚠ [{col}]: {coerced} non-numeric values coerced to NaN")
        coerce_issues += coerced
    else:
        print(f"  ✓ [{col}]: all numeric (dtype={df[col].dtype})")

print(f"\nTotal coercion issues: {coerce_issues}")

Numeric validation for return/metric columns:
  ✓ [return_1yr_pct]: all numeric (dtype=float64)
  ✓ [return_3yr_pct]: all numeric (dtype=float64)
  ✓ [return_5yr_pct]: all numeric (dtype=float64)
  ✓ [benchmark_3yr_pct]: all numeric (dtype=float64)
  ✓ [alpha]: all numeric (dtype=float64)
  ✓ [beta]: all numeric (dtype=float64)
  ✓ [sharpe_ratio]: all numeric (dtype=float64)
  ✓ [sortino_ratio]: all numeric (dtype=float64)
  ✓ [std_dev_ann_pct]: all numeric (dtype=float64)
  ✓ [max_drawdown_pct]: all numeric (dtype=float64)
  ✓ [aum_crore]: all numeric (dtype=int64)
  ✓ [expense_ratio_pct]: all numeric (dtype=float64)

Total coercion issues: 0


## Step 3 — Flag Schemes with Negative Sharpe Ratio

In [4]:
neg_sharpe_mask = df['sharpe_ratio'] < 0
neg_sharpe_count = neg_sharpe_mask.sum()

print(f"Schemes with negative Sharpe ratio: {neg_sharpe_count}")

# Add a flag column for transparency — do NOT drop (negative Sharpe is a valid data point)
df['negative_sharpe_flag'] = neg_sharpe_mask

if neg_sharpe_count > 0:
    print("\nFlagged schemes:")
    flagged = df[neg_sharpe_mask][['amfi_code', 'scheme_name', 'sharpe_ratio']]
    print(flagged.to_string(index=False))
    # Save separate report
    flagged.to_csv(PROCESSED_DIR / 'flagged_negative_sharpe.csv', index=False)
    print(f"\nFlagged rows saved → data/processed/flagged_negative_sharpe.csv")
else:
    print("No negative Sharpe ratios — all schemes are well-performing.")

print(f"\nSharpe ratio stats:")
print(df['sharpe_ratio'].describe().round(4))

Schemes with negative Sharpe ratio: 0
No negative Sharpe ratios — all schemes are well-performing.

Sharpe ratio stats:
count    40.0000
mean      1.3618
std       1.4758
min       0.8000
25%       0.8650
50%       0.9250
75%       0.9850
max       7.6800
Name: sharpe_ratio, dtype: float64


## Step 4 — Validate expense_ratio Range (0.1% – 2.5%)

In [5]:
EXPENSE_MIN = 0.1
EXPENSE_MAX = 2.5

out_of_range_mask = (
    (df['expense_ratio_pct'] < EXPENSE_MIN) |
    (df['expense_ratio_pct'] > EXPENSE_MAX)
)
out_of_range_count = out_of_range_mask.sum()

print(f"expense_ratio range: {df['expense_ratio_pct'].min():.2f}% — {df['expense_ratio_pct'].max():.2f}%")
print(f"Schemes outside SEBI limit (0.1%–2.5%): {out_of_range_count}")

# Flag only — out-of-range expense ratios may need regulatory review, NOT dropped
df['expense_ratio_out_of_range_flag'] = out_of_range_mask

if out_of_range_count > 0:
    print("\nFlagged schemes:")
    oor = df[out_of_range_mask][['amfi_code', 'scheme_name', 'expense_ratio_pct']]
    print(oor.to_string(index=False))
    oor.to_csv(PROCESSED_DIR / 'flagged_expense_ratio.csv', index=False)
    print(f"\nFlagged rows saved → data/processed/flagged_expense_ratio.csv")
else:
    print("All expense ratios within the valid 0.1%–2.5% SEBI range.")

print(f"\nexpense_ratio_pct stats:")
print(df['expense_ratio_pct'].describe().round(4))

expense_ratio range: 0.55% — 1.64%
Schemes outside SEBI limit (0.1%–2.5%): 0
All expense ratios within the valid 0.1%–2.5% SEBI range.

expense_ratio_pct stats:
count    40.0000
mean      1.2370
std       0.3866
min       0.5500
25%       0.7875
50%       1.4250
75%       1.5400
max       1.6400
Name: expense_ratio_pct, dtype: float64


## Step 5 — Remove Duplicates

In [6]:
dups = df.duplicated(subset=['amfi_code']).sum()
print(f"Duplicate amfi_code rows: {dups}")

if dups:
    df.drop_duplicates(subset=['amfi_code'], keep='first', inplace=True)
    df.reset_index(drop=True, inplace=True)
    print(f"Shape after dedup: {df.shape}")
else:
    print("No duplicates found — nothing to drop.")

Duplicate amfi_code rows: 0
No duplicates found — nothing to drop.


## Step 6 — Save Clean Dataset

In [7]:
df_clean = df.copy()
df_clean.to_csv(OUTPUT_PATH, index=False)

print(f"Saved → {OUTPUT_PATH}")
print(f"Final row count : {len(df_clean)}")
print(f"Columns         : {df_clean.columns.tolist()}")

Saved → ../data/processed/clean_performance.csv
Final row count : 40
Columns         : ['amfi_code', 'scheme_name', 'fund_house', 'category', 'plan', 'return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct', 'benchmark_3yr_pct', 'alpha', 'beta', 'sharpe_ratio', 'sortino_ratio', 'std_dev_ann_pct', 'max_drawdown_pct', 'aum_crore', 'expense_ratio_pct', 'morningstar_rating', 'risk_grade', 'negative_sharpe_flag', 'expense_ratio_out_of_range_flag']


## Step 7 — Verification Summary

In [8]:
df_verify = pd.read_csv(OUTPUT_PATH)

print("=" * 55)
print("  CLEAN PERFORMANCE — VERIFICATION REPORT")
print("=" * 55)

print("\n[1] Shape            :", df_verify.shape)

print("\n[2] Null counts")
print(df_verify.isnull().sum())

print("\n[3] Duplicate amfi_code :", df_verify.duplicated(subset=['amfi_code']).sum())
assert df_verify.duplicated(subset=['amfi_code']).sum() == 0, "ERROR: duplicate amfi_codes!"

print("\n[4] Return columns — all numeric?")
for col in ['return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct', 'sharpe_ratio', 'expense_ratio_pct']:
    print(f"    {col}: {df_verify[col].dtype} | nulls={df_verify[col].isna().sum()}")

print(f"\n[5] Negative Sharpe count    : {(df_verify['sharpe_ratio'] < 0).sum()}")
print(f"[6] Expense ratio range      : {df_verify['expense_ratio_pct'].min():.2f}% – {df_verify['expense_ratio_pct'].max():.2f}%")
print(f"[7] negative_sharpe_flag col : {'negative_sharpe_flag' in df_verify.columns}")
print(f"[8] expense_ratio_flag col   : {'expense_ratio_out_of_range_flag' in df_verify.columns}")

print("\n" + "=" * 55)
print("  ALL CHECKS PASSED — clean_performance.csv is ready")
print("=" * 55)

  CLEAN PERFORMANCE — VERIFICATION REPORT

[1] Shape            : (40, 21)

[2] Null counts
amfi_code                          0
scheme_name                        0
fund_house                         0
category                           0
plan                               0
return_1yr_pct                     0
return_3yr_pct                     0
return_5yr_pct                     0
benchmark_3yr_pct                  0
alpha                              0
beta                               0
sharpe_ratio                       0
sortino_ratio                      0
std_dev_ann_pct                    0
max_drawdown_pct                   0
aum_crore                          0
expense_ratio_pct                  0
morningstar_rating                 0
risk_grade                         0
negative_sharpe_flag               0
expense_ratio_out_of_range_flag    0
dtype: int64

[3] Duplicate amfi_code : 0

[4] Return columns — all numeric?
    return_1yr_pct: float64 | nulls=0
    return_3yr_p